# 003 · Dictionaries

> **Syntax › Collections** · Python 3.14

A `dict` maps hashable **keys** to **values**. It is Python's built-in hash map — the equivalent of C#'s `Dictionary<TKey, TValue>` — and it is everywhere: keyword arguments, object attributes, JSON, configuration.

**In this notebook you will learn:**

- how to create dicts and read values safely (`[]` vs `.get()` vs `TryGetValue`)
- adding, updating and removing entries, including `pop` and `setdefault`
- iterating keys, values and items — and why views are *live*
- merging with `|` and `|=`, and the guaranteed insertion order
- `defaultdict` and `Counter` from the `collections` module

## C# mental model

| C# | Python |
|---|---|
| `Dictionary<string, int>` | `dict[str, int]` (type hint only) |
| `new() { ["a"] = 1 }` | `{"a": 1}` |
| `dict.Count` | `len(d)` |
| `dict.ContainsKey(k)` | `k in d` |
| `dict.TryGetValue(k, out var v)` | `d.get(k)` (returns `None` if missing) |
| `dict.Keys` / `dict.Values` | `d.keys()` / `d.values()` |
| `foreach (var (k, v) in dict)` | `for k, v in d.items():` |

Like `List<T>`, a Python dict holds references to any type of key or value — the type hints are not enforced at runtime.

## Creating dicts

```csharp
// C#
var ages = new Dictionary<string, int>
{
    ["alice"] = 30,
    ["bob"] = 25,
};
var empty = new Dictionary<string, int>();
```

In [ ]:
ages = {"alice": 30, "bob": 25}
empty = {}                                   # an empty dict (NOT an empty set)
from_kwargs = dict(alice=30, bob=25)         # keys must be valid identifiers
from_pairs = dict([("alice", 30), ("bob", 25)])
from_zip = dict(zip(["x", "y"], [1, 2]))
from_keys = dict.fromkeys(["a", "b", "c"], 0)

print(ages, empty, from_kwargs, from_pairs, from_zip, from_keys, sep="\n")

## Reading values: `[]` vs `.get()`

`d[key]` raises `KeyError` if the key is missing — the same as C#'s indexer throwing `KeyNotFoundException`. For a "try" lookup, use `.get(key, default)`, which never raises.

```csharp
// C#
var a = ages["alice"];                        // throws if missing
if (ages.TryGetValue("carol", out var c)) { ... }
var d = ages.GetValueOrDefault("carol", -1);
```

In [ ]:
ages = {"alice": 30, "bob": 25}

print(ages["alice"])
print(ages.get("carol"))          # None  — like TryGetValue returning false
print(ages.get("carol", -1))      # -1    — like GetValueOrDefault(key, -1)

try:
    ages["carol"]
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

A pattern that reads like `TryGetValue` uses the walrus operator `:=` to assign and test in one expression:

In [ ]:
if (age := ages.get("bob")) is not None:
    print(f"bob is {age}")

⚠️ **Gotcha:** `if ages.get(k):` is *not* the same as "key exists" — a stored value of `0`, `""` or `None` is falsy. Use `k in d` to test membership *(see Data Types › 004 · Booleans and Truthiness)*.

In [ ]:
stock = {"apples": 0}
print(bool(stock.get("apples")))   # False — but the key exists!
print("apples" in stock)           # True  — the correct check (C#: ContainsKey)

## Adding, updating and removing

| C# `Dictionary` | Python `dict` |
|---|---|
| `d[k] = v` | `d[k] = v` (add or overwrite) |
| `d.Add(k, v)` (throws if exists) | no direct equivalent — check `k in d` first |
| `d.TryAdd(k, v)` | `d.setdefault(k, v)` |
| `d.Remove(k)` | `del d[k]` (raises `KeyError`) |
| `d.Remove(k, out var v)` | `v = d.pop(k)` |
| *(n/a)* | `d.pop(k, default)` — no error if missing |
| `d.Clear()` | `d.clear()` |

In [ ]:
inventory = {"apples": 5}

inventory["pears"] = 3              # add
inventory["apples"] = 10            # overwrite
inventory.update({"plums": 7, "pears": 4})   # bulk add/overwrite
inventory.update(kiwis=2)                     # keyword form
print(inventory)

del inventory["plums"]
removed = inventory.pop("pears")              # remove and return
missing = inventory.pop("mangos", None)       # no KeyError with a default
print(inventory, "| popped:", removed, missing)

`setdefault(k, v)` returns the existing value if `k` is present; otherwise it inserts `v` and returns it. It's handy for "get or create" — but see `defaultdict` below for a cleaner option.

In [ ]:
groups = {}
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    groups.setdefault(word[0], []).append(word)
print(groups)

## Insertion order is guaranteed

Since Python 3.7, a dict **remembers insertion order** as a language guarantee. Iteration always yields keys in the order they were first inserted; overwriting a value keeps the key's original position.

C#'s `Dictionary<TKey, TValue>` makes **no** ordering promise (it often *looks* ordered until you remove items). If you need order in .NET you reach for `SortedDictionary` (sorted by key — different!) or `OrderedDictionary`.

In [ ]:
d = {}
d["zebra"] = 1
d["apple"] = 2
d["mango"] = 3
d["zebra"] = 99        # update — position unchanged
print(list(d))         # ['zebra', 'apple', 'mango']

del d["zebra"]
d["zebra"] = 1         # delete + re-insert moves it to the end
print(list(d))

## Iterating keys, values and items

Iterating a dict directly yields its **keys**. Use `.values()` for values and `.items()` for `(key, value)` pairs — the equivalent of iterating `KeyValuePair<TKey, TValue>` and deconstructing it.

```csharp
// C#
foreach (var (name, score) in scores)
    Console.WriteLine($"{name}: {score}");
```

In [ ]:
scores = {"alice": 92, "bob": 78, "carol": 85}

for name in scores:                     # keys
    print(name, end=" ")
print()

print(sum(scores.values()))             # values

for name, score in scores.items():      # key/value pairs
    print(f"{name}: {score}")

⚠️ **Gotcha:** like C# (`InvalidOperationException: Collection was modified`), you may not add or remove keys while iterating. Iterate over a **copy** of the keys instead:

In [ ]:
scores = {"alice": 92, "bob": 78, "carol": 85}

try:
    for name in scores:
        if scores[name] < 80:
            del scores[name]
except RuntimeError as e:
    print(f"{type(e).__name__}: {e}")

scores = {"alice": 92, "bob": 78, "carol": 85}
for name in list(scores):               # snapshot of the keys
    if scores[name] < 80:
        del scores[name]
print(scores)

## Views are live

`keys()`, `values()` and `items()` return **view** objects, not copies. A view reflects later changes to the dict — similar to how C#'s `Dictionary.KeyCollection` is a live wrapper over the dictionary.

Key views (and item views) also behave like **sets**, so you can use set operators on them.

In [ ]:
prices = {"tea": 3, "coffee": 4}
keys = prices.keys()
print(keys)

prices["juice"] = 5
print(keys)                              # sees the new key

other = {"coffee": 0, "water": 0}
print(prices.keys() & other.keys())      # keys in both  — {'coffee'}
print(prices.keys() - other.keys())      # keys only in prices

## Merging: `|` and `|=`

Python 3.9 added merge operators. `a | b` returns a **new** dict; `a |= b` updates `a` in place. When keys collide, the **right-hand** side wins. The older spelling `{**a, **b}` does the same *(see Collections › 006 · Unpacking)*.

```csharp
// C# — no built-in merge operator; typically:
var merged = new Dictionary<string, string>(defaults);
foreach (var (k, v) in overrides) merged[k] = v;
```

In [ ]:
defaults = {"theme": "light", "lang": "en", "page_size": 20}
overrides = {"theme": "dark", "page_size": 50}

merged = defaults | overrides       # new dict, right wins
print(merged)
print(defaults)                     # unchanged

defaults |= overrides               # in place, like update()
print(defaults)

## Keys must be hashable

Keys must be **hashable** — in practice, immutable: `str`, `int`, `float`, `bool`, `tuple` (of hashables), `frozenset`, and most custom objects. Lists, dicts and sets cannot be keys. This mirrors C#'s requirement that keys implement a stable `GetHashCode`/`Equals` — Python just enforces it at runtime.

In [ ]:
locations = {(51.5, -0.1): "city A", (40.7, -74.0): "city B"}   # tuple keys are fine
print(locations[(51.5, -0.1)])

try:
    bad = {["a", "b"]: 1}
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `1`, `1.0` and `True` are equal and have the same hash, so they are the **same key**:

In [ ]:
d = {1: "int"}
d[1.0] = "float"
d[True] = "bool"
print(d)     # {1: 'bool'} — one key; the first key object is kept, the value is overwritten

## Equality

`==` compares dicts **by content** (same keys mapping to equal values). Order does **not** matter for equality. In C#, `==` on two `Dictionary` instances compares references.

In [ ]:
a = {"x": 1, "y": 2}
b = {"y": 2, "x": 1}
print(a == b)     # True — order is ignored
print(a is b)     # False

## `collections.defaultdict`

A `defaultdict` calls a factory to create a value whenever a missing key is **read**. It removes the need for `setdefault` or "if not in, add" boilerplate. There's no built-in C# equivalent — you'd write a `GetOrAdd` helper or use `CollectionsMarshal.GetValueRefOrAddDefault`.

In [ ]:
from collections import defaultdict

by_letter = defaultdict(list)            # factory: list() -> []
for word in ["apple", "avocado", "banana", "cherry", "cranberry"]:
    by_letter[word[0]].append(word)      # no KeyError on first access

print(dict(by_letter))

word_lengths = defaultdict(int)          # int() -> 0
for word in ["a", "bb", "cc", "ddd"]:
    word_lengths[len(word)] += 1
print(dict(word_lengths))

⚠️ **Gotcha:** *reading* a missing key on a `defaultdict` **inserts** it. Use `in` or `.get()` if you only want to check.

In [ ]:
dd = defaultdict(list)
_ = dd["ghost"]      # just looking...
print(dict(dd))      # {'ghost': []} — it was created
print("other" in dd) # False — `in` doesn't insert

## `collections.Counter`

`Counter` is a dict subclass specialised for counting hashable items — like LINQ `GroupBy(x => x).ToDictionary(g => g.Key, g => g.Count())` in one call.

In [ ]:
from collections import Counter

votes = ["red", "blue", "red", "green", "blue", "red"]
tally = Counter(votes)
print(tally)
print(tally["red"], tally["purple"])     # missing keys count as 0 (no insertion)
print(tally.most_common(2))              # top 2 — like OrderByDescending(...).Take(2)

## Type hints

In [ ]:
config: dict[str, int | str] = {"port": 8080, "host": "localhost"}
config["debug"] = [True]       # runs fine — hints are not enforced
print(config)

## Summary

| Task | C# | Python |
|---|---|---|
| Create | `new Dictionary<K, V> { [k] = v }` | `{k: v}` |
| Read (throws) | `d[k]` → `KeyNotFoundException` | `d[k]` → `KeyError` |
| Try read | `TryGetValue` / `GetValueOrDefault` | `d.get(k, default)` |
| Contains key | `ContainsKey(k)` | `k in d` |
| Add / overwrite | `d[k] = v` | `d[k] = v` |
| Add if absent | `TryAdd(k, v)` | `d.setdefault(k, v)` |
| Remove | `Remove(k)` | `del d[k]` / `d.pop(k)` |
| Iterate pairs | `foreach (var (k, v) in d)` | `for k, v in d.items():` |
| Merge | manual loop | `a \| b`, `a \|= b` |
| Order | not guaranteed | insertion order guaranteed |
| Value equality | manual | `a == b` |
| Grouping helper | `GetOrAdd` helper | `defaultdict(list)` |
| Counting | `GroupBy(...).ToDictionary(...)` | `Counter(items)` |

## Exercises

1. Given `text = "the quick brown fox jumps over the lazy dog the end"`, count word frequencies **without** `Counter` (use `.get()`), then again **with** `Counter`, and print the most common word.
2. Invert a dict: turn `{"a": 1, "b": 2, "c": 1}` into `{1: ["a", "c"], 2: ["b"]}`.
3. Merge `base = {"timeout": 30, "retries": 3}` with `env = {"retries": 5}` so that `env` wins, without modifying `base`.
4. Remove every entry whose value is `None` from `d = {"a": 1, "b": None, "c": 3, "d": None}` **in place**.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
text = "the quick brown fox jumps over the lazy dog the end"
counts = {}
for word in text.split():
    counts[word] = counts.get(word, 0) + 1

from collections import Counter
print(Counter(text.split()).most_common(1))   # [('the', 3)]

# 2
from collections import defaultdict
original = {"a": 1, "b": 2, "c": 1}
inverted = defaultdict(list)
for key, value in original.items():
    inverted[value].append(key)
inverted = dict(inverted)     # {1: ['a', 'c'], 2: ['b']}

# 3
base = {"timeout": 30, "retries": 3}
env = {"retries": 5}
merged = base | env           # {'timeout': 30, 'retries': 5}

# 4 — iterate over a snapshot of the keys
d = {"a": 1, "b": None, "c": 3, "d": None}
for key in list(d):
    if d[key] is None:
        del d[key]
```

</details>